# Load raw — the EL step
Downloads the course dataset from GitHub **directly into a Unity Catalog Volume**
(no local files, no drag-and-drop), then lands it as bronze tables.

First time (To the Cloud): import this notebook (Workspace → Import → URL) and **Run all**.
Every night after (4.01): Task 1 of the Databricks Job, pulled from `scripts/load_raw.py` in your repo.
Safe to rerun: files already in the Volume are skipped, the raw tables are re-landed.

In [0]:
REPO = "lukebarousse/dbt_Analytics_Engineering_Course"
TAG = "dataset-v1"                       # the course dataset release (pinned)
VOLUME = "/Volumes/workspace/jobs/files"

MONTHS = [
    "2025-07", "2025-08", "2025-09", "2025-10", "2025-11", "2025-12",
    "2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06",
]
FILES = [f"raw_job_postings_{m}.parquet" for m in MONTHS] + ["raw_job_skills.parquet"]

In [0]:
# the landing zone: the raw catalog + a schema named for the source system (jobs) + a Volume for the files
# (raw has ONE owner — this notebook; dev and prod both read it)
# spark.sql("CREATE CATALOG IF NOT EXISTS raw")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.jobs")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.jobs.files")

In [0]:
# download from the GitHub release straight into the Volume (~85MB total)
import urllib.request, os

for name in FILES:
    if "postings" in name:
        dest = f"{VOLUME}/job_postings/{name}"
    elif "skills" in name:
        dest = f"{VOLUME}/job_skills/{name}"
    else:
        print(f"  skipping {name} — no matching destination")
        continue
    if os.path.exists(dest):
        print(f"  already have {name}")
        continue
    print(f"  downloading {name} ...")
    urllib.request.urlretrieve(f"https://github.com/{REPO}/releases/download/{TAG}/{name}", dest)

print("Done.")

In [0]:
# land the bronze tables — raw, untouched, exactly as the scraper delivered them
spark.sql(f"""
    CREATE OR REPLACE TABLE workspace.jobs.raw_job_postings AS
    SELECT * FROM read_files('{VOLUME}/job_postings/raw_job_postings_*.parquet', format => 'parquet')
""")
spark.sql(f"""
    CREATE OR REPLACE TABLE workspace.jobs.raw_job_skills AS
    SELECT * FROM read_files('{VOLUME}/job_skills/raw_job_skills.parquet', format => 'parquet')
""")

In [0]:
%skip
# sanity check — expect 843,097 postings and 3,304,574 skill rows
display(spark.sql("""
    SELECT 'raw_job_postings' AS table, COUNT(*) AS rows FROM workspace.jobs.raw_job_postings
    UNION ALL
    SELECT 'raw_job_skills', COUNT(*) FROM workspace.jobs.raw_job_skills
"""))